# Planning, Parallel Thinking, and Multi-Step Tool Use
*How LATS brings tree search into language model reasoning, SPRINT teaches models to think in parallel, and SWiRL trains multi-step tool use that generalizes across entirely different tasks and tools*

# Why Multi-Step Tasks Need More Than Just Reasoning or Acting
Some tasks can't be solved by reasoning alone, or by acting alone — they need both, combined with the ability to **search** across different possible paths and pick the best one.

Take planning a trip as an example:
- **Reasoning:** thinking through what's needed — budget, rough plan, etc.
- **Acting:** gathering more information based on that reasoning — searching the web, reading travel blogs or forums.
- **Search/refinement:** using what was gathered to revise the plan — maybe considering a different destination, or finding new things to do.

Solving a task like this well means cycling through reasoning, acting, and refining a plan, not doing any one of these in isolation.


# LATS: Language Agent Tree Search

**Paper:** [arxiv.org/abs/2310.04406](https://arxiv.org/abs/2310.04406)

The problem this paper set out to solve: when an LLM generates and executes a plan, it tends not to explore very many different possible paths — it doesn't naturally diversify the plans or actions it considers. This is still a common limitation today.

## The Core Idea
Bring well-established techniques from reinforcement learning and multi-step planning — specifically **Monte Carlo Tree Search (MCTS)** — into how a language model reasons and acts. On top of that, let the model take in feedback from its environment as it explores, and use that feedback to refine its future search and planning.


# A Simple Walkthrough: Planning a Trip to Hawaii
Given the prompt "plan a trip to Hawaii," the model might consider different initial actions — for example, asking friends who've already been, or reading relevant subreddits.

1. The framework scores each possible action.
2. Based on that score, it decides which action to **expand** further (i.e., explore more deeply) — in this example, say "asking two friends about their opinions" scores highest.
3. It expands from there — asking friend A and friend B — scoring their responses too.
4. Multiple actions **can be executed in parallel**, rather than strictly one at a time.

The overall approach balances **exploration** (trying less-visited paths, in case they turn out better) with **exploitation** (continuing down the path that currently looks best) — this exploration/exploitation balance is exactly what MCTS is designed for.


# How LATS Differs From Earlier Approaches
- **Compared to Math-Shepherd** (which used a verifier to score and guide reasoning steps): LATS doesn't score the reasoning steps themselves — it scores the **outcomes of the actions** the model actually takes in the environment. It also adds a **reflection** step, where the model evaluates how a given trajectory went.
- **Compared to ReAct:** LATS builds on ReAct's idea of interleaving reasoning and acting with environment feedback, but adds explicit **memory**, **reflection**, and much more structured **planning** — specifically, organizing the whole search as a tree, rather than a single linear sequence of thought-act-observe steps.

## The Building Blocks
- **Chain-of-thought:** decompose reasoning into a sequence of steps.
- **Tree expansion:** each action generates new branches, forming a tree that can be searched and planned over.
- **Environment feedback (from ReAct):** incorporate the results of actions back into the search process, to guide what to try next.


# The Six Stages of LATS
LATS runs through six stages in a loop: **selection, expansion, evaluation, simulation, backpropagation, and reflection.**

## Worked Example: Navigating a Maze
Prompt: "Navigate through a maze to reach an exit." Initial observation: "You're in a dimly lit room. There are two doors, one on the left and one on the right."

1. **Selection:** pick a node in the search tree to expand from (using a formula called UCT, covered below).
2. **Expansion:** from the selected node, sample a few different possible actions. In this example: "open the left door," "open the right door," "inspect the room for clues."
3. Each action is actually **executed** in the environment, and the resulting observation gets appended to the context (e.g., "open the left door" → "dark corridor with paintings").
4. **Evaluation:** score how promising the new state is. Two scores are combined:
   - An **LLM-as-judge score:** prompt the model to rate how promising this state looks, from 0 to 1.
   - A **self-consistency score:** if many actions were sampled (say, 50 instead of just 3) and grouped into categories, the more often a particular action type gets chosen, the higher its self-consistency score.
   These two scores are added together to get one overall value for that action/state.
5. **Simulation:** continue expanding from the highest-scoring state (in a simple "greedy" version, always continuing from the current best) — sampling further actions and continuing until reaching a final state: success, failure, or hitting a set compute/step budget.
6. **Backpropagation:** once a full trajectory finishes (success or failure), that result is used to update the scores of the states and actions that were visited along the way, so future rounds of search are better informed.
7. **Reflection:** after backpropagation, the model also generates its own reflection on the trajectory — reasoning in plain language about what led to success or failure. This reflection step has been found to meaningfully improve the overall quality of the approach.


# The Math Behind Selection: UCT (Upper Confidence Bound for Trees)
UCT is the formula used to decide which node to expand next (the "selection" stage) — borrowed directly from the Monte Carlo Tree Search literature. Its purpose: balance **exploitation** (keep expanding whatever currently looks best) against **exploration** (also try less-visited nodes, since they might turn out better once explored further).

The formula combines two parts:
- The current estimated value of the state, **V(s)**.
- An exploration term, weighted by a hyperparameter, based on **how many times the parent node has been visited (Np)** versus **how many times this specific node has been visited (Ns)**.

The intuition: if a node has been visited relatively few times compared to its parent, the exploration term pushes to visit it more. If a node has already been visited a lot relative to its parent, that term shrinks — making UCT naturally favor exploring the less-visited siblings instead.

## Backpropagation Formula
After a trajectory finishes, the value of each visited state gets updated as a running average: the new value combines the *old* value (weighted by how many times that node has been visited so far) with the *new* return from this trajectory, then divides by the updated total visit count. This is exactly how classic Monte Carlo Tree Search backpropagation works, adapted here for language model reasoning trajectories.


# Results
LATS was tested on:
- **HotpotQA** — a multi-hop question-answering dataset, specifically designed so that answering each question requires retrieving information from at least two different Wikipedia pages, forcing a genuinely multi-step process.
- **WebShop** — a simulated online shopping task requiring multiple sequential actions (e.g., finding a "small portable folding desk, already assembled, in a specific color and finish, under a certain price").
- **Programming tasks** — reported separately in the paper.

## Key Findings
- As the number of sampled trajectories increases, performance improves significantly — meaning LATS effectively converts more test-time compute into better final answers.
- The reflection step (the model reasoning about its own trajectory's success or failure) provided a further meaningful boost.
- On **WebShop**, LATS reached results close to human-expert level — without any fine-tuning, purely through test-time search.
- Independently verified numbers from the published paper: LATS reached **94.4% on HumanEval with GPT-4** (a coding benchmark) and an average score of **75.9 on WebShop with GPT-3.5** — outperforming ReAct, Reflexion, chain-of-thought, and Tree-of-Thought approaches across several of these tasks.


# Strengths and Limitations
**Strengths:** LATS unifies reasoning, acting, and planning into one framework, brings well-established ideas (MCTS) into the language model world, and achieves strong results across several different domains, purely through test-time search — no fine-tuning required. It's also modular and relatively portable to apply to new tasks.

**Limitations:**
- **Cost:** each stage — expansion, evaluation, simulation, backpropagation — adds real computational cost, since it requires many more model calls than a single straight-through generation. The paper doesn't provide a thorough cost-benefit analysis of this trade-off.
- **Irreversible actions:** the approach assumes actions can be explored and backtracked from freely. It doesn't directly address situations where an action might be **irreversible** — for example, an agent that actually executes a financial transaction or a real payment. Exploring and reverting isn't possible in those cases the way it is in a simulated maze or shopping environment, so adapting this kind of tree search to real, consequential actions remains an open challenge.


# Beyond UCT: Other Exploration Strategies
UCT is essentially an upper-confidence-bound approach borrowed from the multi-armed bandit literature (a well-studied area of reinforcement learning about how to balance trying new options versus sticking with what's working). The paper doesn't compare UCT against other bandit-style exploration strategies from that literature — it mainly provides the overall tree-search framework as a platform that others can plug different optimization strategies into.

## Handling Repeated Actions in the Tree
If the same action shows up more than once from a given parent node, the visit count for that action simply gets incremented — this is already captured directly by the UCT formula (via Ns, the visit count for that specific node). This assumes the structure being built really is a **tree**, not a more general graph where the same state could be reached through multiple different paths.


# SPRINT: Teaching Models to Think in Parallel

**Paper:** [arxiv.org/abs/2506.05745](https://arxiv.org/abs/2506.05745)

Modern reasoning models (o1, Gemini 2.5 Pro, DeepSeek-R1, and similar) tend to "think" for longer on harder problems — and this longer thinking is correlated with higher accuracy. As training continues, DeepSeek-R1 gets better at solving hard math problems, and at the same time, its average response length keeps growing — suggesting a real link between generating more thinking tokens and getting better answers.

## The Observation Behind SPRINT
Even though longer reasoning helps, a lot of the individual steps inside a single long reasoning trace are actually **independent of each other** — the model tries alternative approaches, breaks a task into subtasks that don't depend on each other, or verifies earlier steps separately. Much of this could, in principle, happen **at the same time** rather than strictly one step after another — there's no fundamental need to generate all of it sequentially.

## The Core Idea
SPRINT is a post-training and fine-tuning framework that teaches a reasoning model to recognize these parallelization opportunities itself and act on them — alternating between a **planning** role (deciding what needs to be done) and a set of **executor** roles that can run at the same time, tackling independent pieces of the plan in parallel.


# How the Training Data Is Built
Since there wasn't existing data showing a model "thinking in parallel," the authors created it using other LLMs as annotators:

1. Take an existing reasoning model's response to a question (e.g., a DeepSeek-R1 trace).
2. Use another model (e.g., GPT-4o) to break that reasoning trace into discrete **steps**.
3. For each step, have that same model annotate whether it's a **planning** part or an **execution** part — and note that a single plan can sometimes have multiple separate execution pieces underneath it.
4. Using these step annotations, build a **DAG** (a dependency graph) showing which steps depend on which — for example, steps 2 and 4 might both depend only on step 1, but not on each other, meaning they could run in parallel.
5. Use this DAG to **pack** steps into groups that can run at the same time (e.g., step 1 must go first, but steps 2 and 3 can then run in parallel).

The result is a dataset of reasoning traces that have been reorganized and explicitly labeled with which parts are "plan" and which parts are "parallel execution." This dataset is then used to **fine-tune** a reasoning model (in the paper: DeepSeek-R1-Distill-Qwen-7B) so it learns to generate its own reasoning in this same interleaved plan-then-parallel-execute style — using special tags to mark each plan and its corresponding parallel execution steps.


# Why This Matters: Inference Cost and Latency
Multi-step reasoning is expensive and slow — every extra sequential token adds to both the time a person waits for an answer and the compute cost of generating it.

**Normal sequential reasoning:** plan 1 → execute plan 1 → plan 2 → execute plan 2 → ... — entirely one step after another.

**After SPRINT fine-tuning:** if plan 1 and plan 2 turn out to be independent, the model can generate *both* plans first, then execute them **at the same time** — including potentially using external tools (e.g., a calculator or Python) to carry out each plan's execution in parallel — before moving on to the next stage.


# Does the Model Still Know How to Revise Its Own Plans?
If the model realizes partway through that an earlier plan was wrong, can it still go back and revise it, even though it's now been trained to think in parallel? Yes — the model still has access to the full context of everything generated so far, so it can revise or reconsider a plan at any point. What's changed through this fine-tuning isn't the model's ability to backtrack — it's that the model has learned to generate independent plans up front rather than waiting unnecessarily for one plan's execution to finish before even starting to think about the next one, whenever those plans don't actually depend on each other.

## Does This Require Changing the Model's Architecture?
No — the model still predicts one token at a time, exactly as before. The difference is purely in the **structure of what it outputs**: instead of writing "plan, then execution, then plan, then execution" all in one linear sequence, it learns to output something like "here is plan 1 and plan 2" together, tagged clearly, so the surrounding system knows it can branch out and run both plans' executions in parallel.


# Training Recipe and Results
**Recipe:** generate 6,000 reasoning trajectories on the MATH dataset, keep only the ones showing the most parallelization potential, and fine-tune DeepSeek-R1-Distill-Qwen-7B on this reformatted data (using standard supervised fine-tuning).

## An Unexpected Bonus: Accuracy Also Improved
The original goal was purely efficiency — reduce the number of sequential tokens needed, to save on cost and latency. But it turned out that encouraging this more structured, parallel way of thinking also **improved accuracy**, not just speed.

## Verified Results (from the published paper)
- Models fine-tuned with SPRINT matched the accuracy of standard reasoning models on complex domains like math, while generating **up to 39% fewer sequential tokens** on problems requiring more than 8,000 output tokens.
- This benefit **transferred to out-of-distribution tasks** the model wasn't specifically trained on: up to a **45% reduction** in sequential tokens on GPQA, and up to a **65% reduction** on Countdown (a different reasoning task), while still matching the accuracy of the fine-tuned reasoning model.

*(Note: a roughly 3.5% accuracy improvement over the base 7B model, and comparisons to a 32B model's efficiency, have also been referenced elsewhere — these specific figures may reflect an earlier or different experimental configuration than the numbers in the final published paper, which primarily emphasizes the token-reduction results above.)*


# More on SPRINT: Generalization and Behavior

## Generalizing Beyond Training
Even though the fine-tuning was done only on the MATH dataset, the model showed both **more parallelism opportunities** and **higher accuracy** on datasets it was never trained on — including Countdown and GPQA Diamond. This suggests that learning to think in parallel is a somewhat general skill, not something narrowly tied to math problems specifically.

## What If Parallel Branches Individually Look Right But Contradict Each Other?
A natural concern: could two independently-generated plans/branches each look locally correct, but produce a final answer that's wrong once combined — similar to how individual steps in a math derivation can each look fine in isolation but still add up to a wrong final result? Since everything generated — whether sequential or parallel — eventually gets folded back into the same shared context, the model still has to produce one synthesized final answer at the end, and is expected to resolve any contradictions between branches at that point. Both purely sequential and parallel-augmented reasoning can, in principle, suffer from this kind of inconsistency — it isn't unique to the parallel case. It also matters that in this setup, the parallel branches are usually different **steps** toward solving one problem, not different competing **approaches** to the same problem.

## Parallelism Changes Over the Course of a Solution
Harder problems tend to need more rounds of iterative planning and execution — fairly intuitive. Less obviously: the model tends to explore with **more parallelism early on**, and gradually **narrows down to fewer, more focused plans** later in a solution — behaving like a broad search early, converging to a single deep path as it approaches the answer.


# Practical Challenges: Load Balancing and Task-Dependence

## Load Balancing Across Parallel Branches
If two "independent" plans are run in parallel, but one takes much longer to execute than the other, the overall speed is bottlenecked by the slower one (a straggler problem — e.g., if step 4 takes far longer than step 2, running them in parallel still costs as long as step 4 alone). One mitigation: if an execution step turns out to be too simple/fast, it can be merged into a larger chunk rather than run as its own separate parallel branch — though fully optimizing this load balancing across stages remains an area for further work.

## Parallelism Is Task-Dependent
How much benefit SPRINT provides depends heavily on how much natural parallelism a given task actually contains — some reasoning tasks are inherently more parallelizable than others (e.g., the ratio of sequential-token savings differs between MATH and GPQA). On MATH specifically, sequential token count was reduced by around 40% while keeping accuracy comparable to baseline methods — a substantial savings, though the exact benefit will vary by task.

**The pattern:** larger savings show up specifically on problems that require **more thinking** to begin with. If a problem's reasoning trace is already short, there's little room for parallelism to help — and in those cases, adding the extra plan/execute structuring can even make things slightly *worse* than a plain sequential baseline. The real benefit shows up on harder problems that need longer, more elaborate reasoning traces.

**A relevant industry data point:** Claude Sonnet 4.5's system card reportedly encourages the model to use tools extensively — on the order of 100+ tool calls — reflecting a broader trend toward reasoning processes that go well beyond the 8,000–10,000 token range this kind of research was originally built around.


# How Inference Actually Works, Step by Step
To be precise about the inference-time process: the model doesn't generate everything (all plans and executions) at once — it still works through the reasoning **sequentially**, but with parallel branches nested inside that sequence.

1. The model generates plan 1 and plan 2 together (since they're independent).
2. As soon as plan 1 is fully generated, execution of plan 1 begins; the same happens for plan 2.
3. Both executions run **at the same time**.
4. Both results are folded back into the shared context.
5. The model then generates the next set of plans (which might again be one or several, depending on what's independent at that point) — and the cycle continues.

**How this matches the training data:** during training, when two plans are independent, they're placed directly next to each other in the training sequence (plan 1 and plan 2 together, followed by execution 1 and execution 2) — so the model learns from data that already reflects this "generate together, then execute together" structure, and it carries that same behavior over at inference time.


# Room for Further Improvement
This version of SPRINT relies on supervised fine-tuning. Reinforcement learning approaches (e.g., GRPO) could plausibly get even more benefit out of this kind of data — RL-based training tends to generalize this sort of learned behavior further than supervised fine-tuning alone. Other open directions include better coordination and overlap of tool use across parallel branches, and getting a more precise read on the real wall-clock speedups this approach delivers in practice, beyond the token-count savings already measured.


# SWiRL: Step-Wise Reinforcement Learning for Multi-Step Tool Use

**Paper:** [arxiv.org/abs/2504.04736](https://arxiv.org/abs/2504.04736)

Many real tasks — answering a multi-hop question with a search engine, solving a math problem, planning a trip, analyzing data — naturally require several rounds of reasoning combined with tool use, not just one. As the number of steps grows, errors from earlier steps can **compound**, making later steps go wrong too.

## Two Specific Problems SWiRL Addresses
1. **Running tools live during training is slow and unreliable.** Tools can fail, time out, or be slow — and training is already expensive, so calling real tools during every training step adds a lot of friction.
2. **Most RL fine-tuning methods (RLHF, RLAIF, RLEF) are built for single-step tasks** — the model generates one final answer, and the reward is based only on whether that final answer is correct. This doesn't give the model fine-grained feedback on *each individual step* of a longer, multi-step process.

## Design Goals
SWiRL aims to teach a model to: solve complex multi-step problems, know **when** to call a tool and how to phrase a good query for it, stay accurate across a chain of steps, recover from its own earlier mistakes, and know when to stop and produce a final answer — all while **avoiding tool calls during the actual RL training process**, and ideally generalizing to new tools and tasks it wasn't specifically trained on.


# How SWiRL Builds Its Training Data
The process starts by generating **synthetic multi-step trajectories** offline, before any RL training happens:

1. Give the model a prompt and access to tools, and let it work through the problem **one step at a time** — at each step, it can reason (write a chain of thought), call a tool, or decide it's ready to give a final answer.
2. After each action, show the model the actual result from the environment (the tool's output), then prompt it again with the full history so far, and let it decide the next step.
3. Repeat until the model produces a final answer — trajectories can end up being anywhere from 1 to 5+ steps long, depending on the problem.
4. For every individual step, use an **LLM-as-judge** to score how good that step was, given everything that came before it.

All of this — generating trajectories and scoring each step — happens **offline**, across many questions in parallel, before RL training ever begins.

## Two Ways to Filter This Data
- **Process filtering:** only keep trajectories where the judge rated **every single step** as good.
- **Outcome filtering:** keep trajectories where the **final answer** was correct, regardless of how individual steps were scored along the way.

These two filtering strategies were compared directly (covered further in the results).


# How Training Actually Works (Without Calling Tools Live)
The clever part of SWiRL: since all the tool-call trajectories were already generated and scored **offline**, the RL training process itself never needs to call a real tool again.

## Worked Example: "Who is older, [Person A] or [Person B]?"
1. **Action 1:** the model decides it needs to search for Person A's age. The judge scores this action.
2. The environment's response (already collected offline) is shown to the model.
3. **Action 2:** the model searches for Person B's age. The judge scores this action too.
4. **Action 3:** given both search results, the model outputs the final answer, wrapped in an answer tag. The judge scores this final step as well.

During actual RL training, the model is shown the prompt plus everything up through a given step (including the environment's already-collected response), and asked to generate the **next** action — but that action is never actually executed against a real tool. The model just gets a reward based on the quality of the proposed action itself, using the pre-collected environment responses already baked into the training data. All the real tool-calling happened once, earlier, during data creation — not during RL fine-tuning.


# Why the Judge Doesn't Need to See the Tool's Output
A natural question: if the judge never sees what a tool actually returned, how can it meaningfully score whether an action was good?

**The key insight:** the judge isn't scoring whether the tool's *output* was correct — it's scoring whether the **query the model generated was a reasonable one to ask**, given the context so far. For example, if the model asks "what is [Person A]'s age?", the judge can recognize this as a sensible, well-formed question to send to a search engine — without needing to know the actual answer that comes back. This is a form of **process-level reward**: judging the reasoning and tool-use decision itself, not the downstream result. Since the tool call's result is already captured in the earlier context (from the offline data generation stage), none of this requires actually running a tool during training.

Notably, the LLM-as-judge itself wasn't specially trained for this — it was simply prompted directly to perform this scoring.


# Results
SWiRL was tested on a mix of multi-step tool-use, question-answering, and math reasoning tasks. Compared to baseline approaches, it improved relative accuracy by:

| Benchmark | Relative accuracy improvement |
|---|---|
| GSM8K (math) | **+21.5%** |
| HotPotQA (multi-hop QA) | **+12.3%** |
| CofCA | **+14.8%** |
| MuSiQue (multi-hop QA) | **+11.1%** |
| BeerQA (multi-hop QA) | **+15.3%** |

**Generalization:** training on just one task (e.g., HotPotQA) improved performance on other, related tasks it wasn't specifically trained on — suggesting the model is learning a genuinely transferable skill (good step-by-step reasoning and tool use), not just memorizing patterns specific to one dataset.


# The Training Objective, in Plain Terms
At a high level, the RL objective optimizes the **expected reward of a single action**, given all the context that came before it (the sequence of prior states and actions so far). This is done separately for each action across a trajectory — action 1's reward is judged given only what came before it, action 2's reward given everything up through action 1, and so on, all the way through the final action.

## What This Looks Like at Inference Time
Once a model has been trained this way, using it looks like an iterative loop:
1. Give the model a question and tell it which tools are available (e.g., "if a calculator would help, generate a query wrapped in these tags").
2. The model responds — possibly by calling a tool (e.g., proposing a calculation).
3. The tool actually runs this time (since this is real inference, not training), and its result is shown back to the model.
4. The model is prompted again with the full history so far, and decides on the next action — another tool call, or a final answer (wrapped in an answer tag) if it has enough information.
5. This repeats until the model outputs a final answer.


# Experimental Setup
- **Data generation model:** Gemma-2-27B was used to create the synthetic multi-step training data.
- **Source questions:** drawn from HotPotQA and GSM8K, totaling around 50,000 examples.
- **Filtering strategies compared:** process-filtered only, outcome-filtered only, both process-and-outcome filtered, and random (unfiltered) data.


# Why Process-Filtered Data Worked Best
A somewhat counterintuitive result: training only on **process-filtered** data (steps that an LLM-as-judge rated as reasonable, regardless of whether the final answer was correct) worked **better** than being strict and only keeping trajectories where the final outcome was also correct.

**Why this makes sense on reflection:** if training data is restricted to only trajectories where the model already got the right final answer, that data mostly reinforces problems the model could already solve — it doesn't teach the model much about handling problems it previously *couldn't* solve, even if some of the individual steps along the way were reasonable. Process-filtered data casts a wider net, including trajectories with good step-by-step reasoning even when the final answer didn't land — giving the model more opportunity to learn *how to think* through a problem, not just to repeat trajectories that already worked.


# The Standout Result: Generalizing Across Tools and Domains
The model was trained on GSM8K math problems, learning to use SymPy (a symbolic math/calculator tool). It was then tested on a completely different task and tool:

- Tested on **HotPotQA** (using a **search tool**, not a calculator) — accuracy improved from **65 to 73**.
- As a general baseline comparison, going from the base model to the GSM8K-trained model also showed a smaller but real improvement (65 to 71) even without directly training on the target task.

The reverse also held: training on HotPotQA with a search tool improved performance on GSM8K using a completely different tool (Python/SymPy). Neither of these target datasets was part of the original training data.

**What this suggests:** the model isn't just learning to use one specific tool well — it's learning a more general skill of **how to reason step by step and how to decide when and how to invoke a tool**, a skill that transfers across both different tasks and different tools entirely.

## More Training Data Helps, Even Out-of-Domain
Scaling the synthetic training data from about 100 examples up to 10,000 (training only on HotPotQA with a search tool) continued to improve performance not just on HotPotQA itself, but also on a completely different, untrained-on task: MATH and GSM8K problems. This is strong evidence that generating synthetic multi-step data in one easy-to-simulate environment can generalize to entirely different tools and domains — suggesting this approach could scale well if pushed further.


# Why Does This Work? Looking at Process Correctness
Measuring the average process reward per step (i.e., how good the model's individual reasoning/tool-use steps are, not just the final answer) before and after this RL fine-tuning: the model's step-by-step reasoning quality clearly improved — both **in-distribution** (on HotPotQA, which it was trained on) and **out-of-distribution** (on GSM8K, which it wasn't trained on). This supports the idea that the model is getting genuinely better at multi-step thinking itself, not just getting lucky on specific benchmarks.


# RL vs. Supervised Fine-Tuning
The same overall idea — training on these multi-step trajectories — could also be done with plain supervised fine-tuning instead of RL. **Multi-step RL clearly outperformed supervised fine-tuning by a meaningful margin.**

Interestingly, supervised fine-tuning behaved the opposite way process-filtering did for RL: SFT worked best when trained only on trajectories where **both** the process and the final outcome were correct — using only process-filtered (but not necessarily outcome-correct) data actually hurt SFT performance.

**Why the difference:** supervised fine-tuning is essentially imitation learning — the model is shown a trajectory and directly encouraged to reproduce it. If the final outcome in that trajectory was wrong, imitating it can actively hurt performance. RL doesn't have this problem in the same way — the model gets to try a **new** action given the prior context, and is rewarded based on that specific new attempt, giving it a chance to improve beyond simply copying potentially-flawed training trajectories.


# Summary
- SWiRL generalizes well across both **different datasets** and **different tools** — training on one task/tool combination transfers meaningfully to entirely different ones.
- The model learns better from **process-filtered** data than from strictly outcome-correct data.
- Performance keeps improving with more synthetic training data, both in-domain and out-of-domain.
- The underlying mechanism appears to be a genuine improvement in **step-by-step reasoning quality** (process correctness), not just better performance on specific benchmarks — and RL clearly outperforms plain supervised fine-tuning on this kind of multi-step data.
